In [4]:
import os
from docent.data_models.agent_run import AgentRun
from docent._loader.load_inspect import load_inspect_eval
from docent.sdk.client import DocentClient
os.environ['ENV_TYPE'] = 'swe-bench'
# from docent._loader.load_inspect import load_swebench

client = DocentClient(server_url="http://localhost:8889", web_url="http://localhost:4001", email="sherwinxyu@gmail.com")
fgs = client.list_framegrids()
print(fgs)


if fgs:
    fg_id = fgs[0]['id']
    print('fg', fg_id, fgs[0]['name'])
else:
    fg_id = client.create_framegrid(name="sherwin-framegrid")

SWE_BENCH_LOGS: dict[str, str | tuple[str, dict[str, str]]] = {
    # "swebench-sonnet37-old": f"/home/ubuntu/artifacts/vincent/swe_bench_logs/2025-04-09T21-09-59+00-00_swe-bench_8AcW4AHxbhgtoqEbe5FQcT.eval",
    # "swebench-sonnet37-new": f"/home/ubuntu/artifacts/vincent/swe_bench_logs/2025-04-10T21-39-15+00-00_swe-bench_TZrCQjagGBxzuSrXnE3fqj.eval",
    "swebench-sonnet35-tools": "~/projects/docent-artifacts/eval1.eval",
    "swebench-sonnet37-tools": "~/projects/docent-artifacts/eval2.eval",
}
def load_swebench() -> list[AgentRun]:
    return load_inspect_eval(SWE_BENCH_LOGS)

TRANSCRIPTS = load_swebench()
client.add_agent_runs(fg_id, TRANSCRIPTS)

[{'id': 'be1ba65c-f3fd-4e11-8cad-2acf94e9b243', 'name': 'sherwin-framegrid', 'description': None, 'created_by': '910dc519-0aa2-437c-98e3-3c9f21c862e5', 'created_at': '2025-06-09T21:15:10.659484'}]
fg be1ba65c-f3fd-4e11-8cad-2acf94e9b243 sherwin-framegrid
14:17:33 [INFO] docent._loader.load_inspect: Loading swebench-sonnet35-tools from ~/projects/docent-artifacts/eval1.eval
14:17:34 [INFO] docent._loader.load_inspect: Loading swebench-sonnet37-tools from ~/projects/docent-artifacts/eval2.eval
14:17:42 [INFO] docent.sdk.client: Successfully added 98 agent runs to FrameGrid 'be1ba65c-f3fd-4e11-8cad-2acf94e9b243'


In [ ]:
import json

from docent._loader.load_inspect import InspectAgentRunMetadata
from docent._log_util import get_logger
from docent.data_models.agent_run import AgentRun
from docent.data_models.chat import ChatMessage, parse_chat_message
from docent.data_models.transcript import Transcript

root = "/Users/sherwin/projects/docent-artifacts/o3_4o_transcripts"
O3_LOGS = {
    f"{root}/ad14cf0e.json": "Time_elapsed",
    f"{root}/d731afd7.json": "Random_seed_1",
    f"{root}/83496f36.json": "MacBook_Pro",
    f"{root}/b21efca0.json": "Yap_score_2",
    f"{root}/0b898987.json": "What_time_is_it_1",
    f"{root}/6abbb2c5.json": "Generating_a_random_prime",
    f"{root}/447ccd14.json": "Writing_a_new_poem",
}

MINI_4O_LOGS = {
    f"{root}/ad14cf0e_2.json": "Time_elapsed",
    f"{root}/d731afd7_2.json": "Random_seed_1",
    f"{root}/83496f36_2.json": "MacBook_Pro",
    f"{root}/b21efca0_2.json": "Yap_score_2",
    f"{root}/0b898987_2.json": "What_time_is_it_1",
    f"{root}/6abbb2c5_2.json": "Generating_a_random_prime",
    f"{root}/447ccd14_2.json": "Writing_a_new_poem",
}


def load_o3() -> list[AgentRun]:
    print("Loading o3")
    transcripts: list[AgentRun] = []
    for path, name in O3_LOGS.items():
        with open(path, "r") as f:
            sample = json.load(f)

        messages: list[ChatMessage] = []

        for message_data in sample["messages"]:
            chat_message = parse_chat_message(message_data)
            messages.append(chat_message)

        metadata = InspectAgentRunMetadata(
            epoch_id=0,
            experiment_id="human-generated_attacks",
            intervention_description=None,
            intervention_index=None,
            intervention_timestamp=None,
            model="o3-2025-04-03",
            task_args={},
            is_loading_messages=False,
            task_id="",
            sample_id=name,
            original_sample_id_type="str",
            scores={"": False},
            default_score_key="",
            scoring_metadata={},
            additional_metadata={},
        )
        transcript = AgentRun(
            transcripts={"default": Transcript(messages=messages)},
            metadata=metadata,
        )
        transcripts.append(transcript)

    for path, name in MINI_4O_LOGS.items():
        try:
            with open(path, "r") as f:
                sample = json.load(f)
        except:
            raise Exception(f"Error loading {path}")

        messages: list[ChatMessage] = []

        for message_data in sample["messages"]:
            chat_message = parse_chat_message(message_data)
            messages.append(chat_message)

        metadata = InspectAgentRunMetadata(
            epoch_id=0,
            experiment_id="human-generated_attacks_2",
            intervention_description=None,
            intervention_index=None,
            intervention_timestamp=None,
            model="gpt-4o-mini-2024-07-18",
            task_args={},
            is_loading_messages=False,
            task_id="",
            sample_id=name,
            original_sample_id_type="str",
            scores={"": False},
            default_score_key="",
            scoring_metadata={},
            additional_metadata={},
        )
        transcript = AgentRun(
            transcripts={"default": Transcript(messages=messages)},
            metadata=metadata,
        )
        transcripts.append(transcript)

    return transcripts
